# Titanic — The Final Boss (Group Survival)
**Goal:** Reach 85%+ accuracy using the **Woman-Child-Group (WCG)** effect.

### The Logic:
1.  Passengers traveled in families/groups (same Surname + same Ticket).
2.  For women and children in a group, their survival is highly correlated.
3.  We will find these groups and create a `GroupSurvival` signal.

## 1. Setup & Data Loading

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import cross_val_score
from sklearn.preprocessing import LabelEncoder

df = pd.read_csv('Titanic-Dataset.csv')
df_orig = df.copy()

## 2. Extract Surname & Ticket Groups

In [ ]:
# 1. Extract Surname
df['Surname'] = df['Name'].apply(lambda x: x.split(',')[0].lower())

# 2. Identify Groups (Same Ticket or Same Surname + Fare)
# We use Ticket as the primary group ID
df['GroupID'] = df['Ticket']

print('Total Passengers:', len(df))
print('Unique Groups (by Ticket):', df['GroupID'].nunique())

## 3. The Woman-Child-Group (WCG) Feature

In [ ]:
# Create a flag for 'Woman or Child' (Age < 13 or Sex = female)
df['IsWomanOrChild'] = ((df['Age'] < 13) | (df['Sex'] == 'female')).astype(int)

# Calculate survival stats for each group (ONLY for women and children)
group_stats = df.groupby('GroupID')['Survived'].agg(['count', 'mean'])
group_stats.columns = ['GroupSize', 'GroupSurvivalRate']

# Filter groups that actually have women/children and more than 1 person
df = df.merge(group_stats, on='GroupID', how='left')

# THE MAGIC FEATURE:
# If you are in a group where others survived -> 1
# If you are in a group where others died -> 0
# If you are alone or no data -> 0.5 (Neutral)

df['GroupSignal'] = 0.5 # Default neutral

for i, row in df.iterrows():
    if row['GroupSize'] > 1:
        # Look at the REST of the group (excluding this person)
        others = df[(df['GroupID'] == row['GroupID']) & (df.index != i)]
        if others['Survived'].max() == 1.0:
            df.at[i, 'GroupSignal'] = 1
        elif others['Survived'].min() == 0.0:
            df.at[i, 'GroupSignal'] = 0

print('GroupSignal Distribution:\n', df['GroupSignal'].value_counts())

## 4. Final Feature Set

In [ ]:
# Bring in the best features from 05_advanced_modeling
df['Title'] = df['Name'].str.extract(' ([A-Za-z]+)\.', expand=False)
df['Title'] = df['Title'].replace(['Lady', 'Countess','Capt', 'Col','Don', 'Dr', 'Major', 'Rev', 'Sir', 'Jonkheer', 'Dona'], 'Other')
df['Title'] = df['Title'].replace(['Mlle', 'Ms'], 'Miss').replace('Mme', 'Mrs')

df['Age'] = df.groupby(['Pclass', 'Title'])['Age'].transform(lambda x: x.fillna(x.median()))
df['Age_Class'] = df['Age'] * df['Pclass']
df['Sex'] = df['Sex'].map({'male': 0, 'female': 1})
df['HasCabin'] = df['Cabin'].notnull().astype(int)

# One-Hot encode only what we need
df = pd.get_dummies(df, columns=['Title', 'Embarked'], drop_first=True)

# Select final numeric columns
cols_to_keep = ['Survived', 'Pclass', 'Sex', 'Age', 'Age_Class', 'SibSp', 'Parch', 
                'Fare', 'HasCabin', 'GroupSignal'] + [c for c in df.columns if 'Title_' in c or 'Embarked_' in c]

final_df = df[cols_to_keep].copy()
final_df = final_df.astype(float)
final_df.head()

## 5. Modeling & Cross-Validation

In [ ]:
X = final_df.drop('Survived', axis=1)
y = final_df['Survived']

rf = RandomForestClassifier(n_estimators=100, max_depth=5, random_state=42)
cv_scores = cross_val_score(rf, X, y, cv=10)

print('Final Boss CV Accuracy:', cv_scores.mean().round(4))
print('Min Score:', cv_scores.min().round(4))
print('Max Score:', cv_scores.max().round(4))

### Conclusion
By using `GroupSignal`, we are capturing the 'Family Effect'. If this score is above 84%, you have reached the practical limit of what can be predicted using this data. 

To go even higher, one would need to use **Boosting algorithms** (XGBoost) or **Stacking**, but `GroupSignal` is the single most powerful feature in the Titanic competition.